In [1]:
# =============================================================
# 🎵 Model A: TF-IDF + SVD Reranking (악기 → 음악 추천)
# =============================================================

import pandas as pd
import numpy as np
import os
import joblib
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import load_npz

# -------------------------------------------------------------
# 1️⃣ 데이터 및 모델 로드
# -------------------------------------------------------------
BASE_DIR = '../'
OUTPUT_DIR = os.path.join(BASE_DIR, 'output')

meta_path = os.path.join(OUTPUT_DIR, 'preprocessed_metadata.pkl')
matrix_path = os.path.join(OUTPUT_DIR, 'tfidf_embedding_matrix.npz')
svd_model_path = os.path.join(OUTPUT_DIR, 'svd_collaborative_instrument2music.joblib')
reviews_path = os.path.join(OUTPUT_DIR, 'preprocessed_reviews.csv')

meta_df = pd.read_pickle(meta_path)
tfidf_matrix = load_npz(matrix_path)
svd_model = joblib.load(svd_model_path)
reviews_df = pd.read_csv(reviews_path, quoting=3, escapechar='\\')

meta_df = meta_df.reset_index(drop=True)
asin_to_index = pd.Series(meta_df.index, index=meta_df["asin"])

print(f"[✓] 메타데이터: {meta_df.shape}, TF-IDF 행렬: {tfidf_matrix.shape}")

# -------------------------------------------------------------
# 2️⃣ TF-IDF 콘텐츠 유사도 함수
# -------------------------------------------------------------
def tfidf_scores_for_instrument(target_asin, meta_df, tfidf_matrix, asin_to_index):
    """
    특정 악기(asin)에 대해 음악(앨범) 도메인과의 TF-IDF 유사도 계산
    """
    if target_asin not in asin_to_index:
        raise ValueError(f"{target_asin} 을(를) 찾을 수 없습니다.")
        
    # 악기 벡터
    target_idx = int(asin_to_index[target_asin])
    target_vec = tfidf_matrix[target_idx]
    
    # 음악 도메인 필터링
    music_mask = np.array(meta_df["domain"] == "music")
    music_matrix = tfidf_matrix[music_mask]
    music_asins = meta_df.loc[music_mask, "asin"].values
    
    # 코사인 유사도 계산
    sims = cosine_similarity(target_vec, music_matrix).flatten()
    return dict(zip(music_asins, sims))  # {asin: similarity}

# -------------------------------------------------------------
# 3️⃣ SVD 예측 점수 캐싱 함수
# -------------------------------------------------------------
def cache_svd_predictions(user_id, meta_df, svd_model):
    """
    해당 사용자가 모든 음악 아이템에 대해 예측한 SVD 평점을 캐싱
    """
    music_asins = meta_df.loc[meta_df["domain"] == "music", "asin"].values
    scores = {}
    for asin in music_asins:
        scores[asin] = svd_model.predict(user_id, asin).est / 5.0  # 0~1 정규화
    return scores

# -------------------------------------------------------------
# 4️⃣ Reranking 추천 함수
# -------------------------------------------------------------
def recommend_rerank_model(user_id, instrument_asin,
                           meta_df, tfidf_matrix, asin_to_index, svd_model,
                           alpha=0.4, beta=0.6, N=10):
    """
    Model A (TF-IDF + SVD Rerank)
    악기 → 음악 추천 (콘텐츠 + 협업 기반)
    """
    # 1. TF-IDF 콘텐츠 점수
    tfidf_scores = tfidf_scores_for_instrument(instrument_asin, meta_df, tfidf_matrix, asin_to_index)
    
    # 2. SVD 예측 점수
    svd_scores = cache_svd_predictions(user_id, meta_df, svd_model)
    
    # 3. 두 점수 결합 (reranking)
    rerank_scores = {}
    for asin, c in tfidf_scores.items():
        s = svd_scores.get(asin, 0.0)
        rerank_scores[asin] = alpha * c + beta * s
    
    # 4. 상위 N개 추천
    top_items = sorted(rerank_scores.items(), key=lambda x: x[1], reverse=True)[:N]
    top_df = pd.DataFrame(top_items, columns=["asin", "rerank_score"])
    
    # 5. 제목 매핑
    top_df = top_df.merge(meta_df[["asin", "title_str"]], on="asin", how="left")
    return top_df

# -------------------------------------------------------------
# 5️⃣ 테스트 실행
# -------------------------------------------------------------
# 예시: 첫 번째 사용자 / 임의 악기 선택
sample_user = reviews_df["reviewerID"].iloc[0]
instrument_items = meta_df.loc[meta_df["domain"] == "instruments", "asin"]

if len(instrument_items) == 0:
    raise ValueError("⚠️ 악기(instruments) 도메인이 없습니다. 전처리 확인 필요.")
instrument_asin = instrument_items.iloc[0]

# 추천 실행
recommendations = recommend_rerank_model(sample_user, instrument_asin,
                                         meta_df, tfidf_matrix, asin_to_index, svd_model,
                                         alpha=0.4, beta=0.6, N=10)

# -------------------------------------------------------------
# 6️⃣ 결과 출력
# -------------------------------------------------------------
print(f"\n[사용자 ID] {sample_user}")
print(f"[기준 악기 ASIN] {instrument_asin}")
print("\n🎯 Model A (TF-IDF + SVD Reranking) 추천 결과\n")
print(recommendations.to_markdown(index=False))

# -------------------------------------------------------------
# 7️⃣ 추천 결과 저장 (05_evaluation.ipynb용)
# -------------------------------------------------------------
save_path = os.path.join(OUTPUT_DIR, 'eval_rerank_results.csv')

# 평가를 위해 reviewerID, asin, rerank_score 포함 저장
recommendations_out = recommendations.copy()
recommendations_out.insert(0, 'reviewerID', sample_user)
recommendations_out.to_csv(save_path, index=False)

print(f"\n💾 평가용 추천 결과 저장 완료: {save_path}")

# -------------------------------------------------------------
# 여러 사용자에 대해 추천 결과 생성 및 저장
# -------------------------------------------------------------
sample_users = reviews_df["reviewerID"].drop_duplicates().head(100)  # 상위 100명만 예시
all_results = []

for user in sample_users:
    instrument_items = meta_df.loc[meta_df["domain"] == "instruments", "asin"]
    if len(instrument_items) == 0:
        continue
    instrument_asin = np.random.choice(instrument_items)
    
    recs = recommend_rerank_model(
        user, instrument_asin,
        meta_df, tfidf_matrix, asin_to_index, svd_model,
        alpha=0.4, beta=0.6, N=10
    )
    recs.insert(0, "reviewerID", user)
    all_results.append(recs)

# 합쳐서 저장
final_recs = pd.concat(all_results, ignore_index=True)
final_recs.to_csv(os.path.join(OUTPUT_DIR, "eval_rerank_results.csv"), index=False)
print(f"💾 여러 사용자 추천 결과 저장 완료: {final_recs.shape}")


[✓] 메타데이터: (367145, 21), TF-IDF 행렬: (367145, 20000)

[사용자 ID] AGKASBHYZPGTEPO6LWZPVJWB2BVA
[기준 악기 ASIN] B01M4HO6RK

🎯 Model A (TF-IDF + SVD Reranking) 추천 결과

| asin       |   rerank_score | title_str           |
|:-----------|---------------:|:--------------------|
| B000IHYWH2 |       0.622524 | Focus 9: New Skin   |
| B00JKJ133U |       0.621596 | Series              |
| B00H9GJSAQ |       0.621596 | Series              |
| B00H9GJP6S |       0.621596 | Series              |
| B00H9GJPZE |       0.621596 | Series              |
| B00JM5CCNM |       0.621596 | Series              |
| B00H9GJSKG |       0.621596 | Series              |
| B00JKJ0XYA |       0.621596 | Series              |
| B00H9GJQW6 |       0.621596 | Series              |
| B00WM93TAK |       0.621596 | Bigbang Made Series |

💾 평가용 추천 결과 저장 완료: ../output/eval_rerank_results.csv
💾 여러 사용자 추천 결과 저장 완료: (1000, 4)
